In [3]:
import sqlite3, pandas as pd
from IPython.display import display

con = sqlite3.connect("data/spacex.sqlite")
TABLE = "spacex"

# Introspect columns once
cols = pd.read_sql(f"PRAGMA table_info({TABLE});", con)
ALL_COLS = [c for c in cols['name']]

def pick(*cands, required=True):
    low = {c.lower(): c for c in ALL_COLS}
    for cand in cands:
        if cand.lower() in low: 
            return low[cand.lower()]
    if required:
        raise KeyError(f"Missing required column. Tried {cands}. Available: {ALL_COLS}")
    return None

COL_SITE   = pick("launch_site","LaunchSite","Site","Launch Site")
COL_PAY    = pick("payload_mass_kg","PayloadMass","PAYLOADMASS__KG_","Payload_Mass","PayloadMass__kg__")
COL_BOOST  = pick("booster_version","BoosterVersion", required=False)  # optional
COL_CLASS  = pick("class","Class","landing_success")
COL_DATE   = pick("date_utc","Date","launch_date")
COL_LOUT   = pick("landing_outcome","Landing_Outcome","LandingOutcome", required=False)  # optional
COL_CUST   = pick("customer","customers","customer_name","mission","mission_name","payload_id", required=False)  # may be None

def show(q, title=None):
    if title: print(f"\n=== {title} ===")
    df = pd.read_sql(q, con)
    try: display(df.style.hide(axis="index"))
    except Exception: display(df)
    return df

print("Detected columns:", {
    "site": COL_SITE, "payload": COL_PAY, "booster": COL_BOOST,
    "class": COL_CLASS, "date": COL_DATE, "landing_outcome": COL_LOUT, "customer_like": COL_CUST
})


Detected columns: {'site': 'launch_site', 'payload': 'payload_mass_kg', 'booster': None, 'class': 'Class', 'date': 'date_utc', 'landing_outcome': None, 'customer_like': None}


In [5]:
import sqlite3
import pandas as pd
from IPython.display import display

# Database setup
CONN = sqlite3.connect("data/spacex.sqlite")
TABLE = "spacex"

# Column aliases
SITE = "launch_site"
PAY = "payload_mass_kg"
ORBIT = "orbit"
DATE = "date_utc"
CLASS = "Class"
YEAR = "year"


def show(query: str, title: str | None = None) -> pd.DataFrame:
    """Run a SQL query against the SpaceX table and display the result nicely."""
    if title:
        print(f"\n=== {title} ===")
    df = pd.read_sql_query(query, CONN)
    try:
        display(df.style.hide(axis="index"))
    except Exception:
        display(df)
    return df


# Unique launch sites
df_sites = show(
    f"""
    SELECT DISTINCT {SITE} AS LaunchSite
    FROM {TABLE}
    ORDER BY LaunchSite;
    """,
    title="Unique Launch Sites",
)

# Example rows from CCA* launch sites
df_cca_example = show(
    f"""
    SELECT *
    FROM {TABLE}
    WHERE {SITE} LIKE 'CCA%'
    LIMIT 5;
    """,
    title="Example Rows: Launch Site starting with 'CCA'",
)

# Total payload across all missions
df_total_payload = show(
    f"""
    SELECT SUM({PAY}) AS TotalPayloadKg
    FROM {TABLE};
    """,
    title="Total Payload (All Missions)",
)
print(
    "Note: The dataset lacks a 'customer' column. If customer data existed, "
    "you could filter by LOWER(customer) LIKE '%nasa%'."
)

# Average payload by orbit
df_avg_payload_by_orbit = show(
    f"""
    SELECT {ORBIT} AS Orbit,
           ROUND(AVG({PAY}), 1) AS AvgPayloadKg,
           COUNT(*) AS Flights
    FROM {TABLE}
    GROUP BY {ORBIT}
    ORDER BY AvgPayloadKg DESC;
    """,
    title="Average Payload by Orbit",
)

# Earliest successful mission date (Class = 1)
df_first_success_date = show(
    f"""
    SELECT MIN({DATE}) AS FirstSuccessfulMissionDate
    FROM {TABLE}
    WHERE {CLASS} = 1;
    """,
    title="First Successful Mission Date",
)

# Successful missions with payload between 4–6t by site & orbit
df_success_4_6t_by_site_orbit = show(
    f"""
    SELECT DISTINCT {SITE} AS LaunchSite,
           {ORBIT}  AS Orbit
    FROM {TABLE}
    WHERE {CLASS} = 1
      AND {PAY} BETWEEN 4000 AND 6000
    ORDER BY LaunchSite, Orbit;
    """,
    title="Successful Missions (4–6 t) by Site & Orbit",
)

# Success vs failure counts (+ percentage)
df_outcomes = show(
    f"""
    SELECT {CLASS} AS Outcome, COUNT(*) AS Missions
    FROM {TABLE}
    GROUP BY {CLASS}
    ORDER BY Outcome DESC;
    """,
    title="Success vs Failure Count",
)
_total = int(df_outcomes["Missions"].sum())
df_outcomes_pct = df_outcomes.assign(
    Percent=(df_outcomes["Missions"] / _total * 100).round(1)
)
try:
    display(df_outcomes_pct.style.hide(axis="index"))
except Exception:
    display(df_outcomes_pct)

# Max-payload mission(s) with site/orbit/date
df_max_payload_missions = show(
    f"""
    SELECT {SITE}  AS LaunchSite,
           {ORBIT} AS Orbit,
           {PAY}   AS PayloadKg,
           {DATE}  AS DateUTC
    FROM {TABLE}
    WHERE {PAY} = (SELECT MAX({PAY}) FROM {TABLE});
    """,
    title="Max Payload Mission(s)",
)

# 2015 failures by launch site
df_failures_2015_by_site = show(
    f"""
    SELECT {SITE} AS LaunchSite,
           COUNT(*) AS Failures_2015
    FROM {TABLE}
    WHERE CAST(strftime('%Y', {DATE}) AS INT) = 2015
      AND {CLASS} = 0
    GROUP BY {SITE}
    ORDER BY Failures_2015 DESC;
    """,
    title="2015 Failures by Launch Site",
)

# Outcomes and orbits in a date window
df_outcomes_window = show(
    f"""
    SELECT {CLASS} AS Outcome, COUNT(*) AS Cnt
    FROM {TABLE}
    WHERE {DATE} BETWEEN '2010-06-04' AND '2017-03-20'
    GROUP BY {CLASS}
    ORDER BY Cnt DESC;
    """,
    title="Outcomes (2010-06-04 → 2017-03-20)",
)

df_orbits_window_stats = show(
    f"""
    SELECT {ORBIT} AS Orbit,
           COUNT(*) AS Flights,
           ROUND(AVG({CLASS}) * 100, 1) AS SuccessRatePct
    FROM {TABLE}
    WHERE {DATE} BETWEEN '2010-06-04' AND '2017-03-20'
    GROUP BY {ORBIT}
    ORDER BY Flights DESC;
    """,
    title="Orbits in Window: Volume & Success Rate",
)



=== Unique Launch Sites ===


LaunchSite
CCSFS SLC 40
KSC LC 39A
Kwajalein Atoll
VAFB SLC 4E



=== Example Rows: Launch Site starting with 'CCA' ===


flight_number,date_utc,year,launch_site,launch_lat,launch_lon,payload_mass_kg,orbit,reuse_count,Class



=== Total Payload (All Missions) ===


...

LaunchSite,Failures_2015
CCSFS SLC 40,5



=== Outcomes (2010-06-04 → 2017-03-20) ===


Outcome,Cnt
0,21
1,11



=== Orbits in Window: Volume & Success Rate ===


Orbit,Flights,SuccessRatePct
GTO,13,23.100000
ISS,10,40.000000
LEO,5,40.000000
PO,3,33.300000
ES-L1,1,100.000000


In [6]:
from pathlib import Path
Path("images").mkdir(exist_ok=True)

def run_sql(q, csv_name):
    df = pd.read_sql_query(q, con)
    display(df.style.hide(axis="index"))
    df.to_csv(f"images/{csv_name}", index=False)
    return df

# 1) Success by site
q_site = """
SELECT LaunchSite, SUM(class) AS Successes, COUNT(*) AS Flights,
       ROUND(100.0*AVG(class),1) AS SuccessRatePct
FROM SPACEXTBL
GROUP BY LaunchSite
ORDER BY SuccessRatePct DESC;
"""
df_site = run_sql(q_site, "sql_site_success.csv")

# 2) Success by booster
q_booster = """
SELECT BoosterVersion, SUM(class) AS Successes, COUNT(*) AS Flights,
       ROUND(100.0*AVG(class),1) AS SuccessRatePct
FROM SPACEXTBL
GROUP BY BoosterVersion
ORDER BY SuccessRatePct DESC
LIMIT 10;
"""
df_booster = run_sql(q_booster, "sql_booster_success.csv")

# 3) Avg payload & success by orbit
q_orbit = """
SELECT Orbit,
       ROUND(AVG(PayloadMass),1) AS AvgPayloadKg,
       ROUND(100.0*AVG(class),1) AS SuccessRatePct
FROM SPACEXTBL
GROUP BY Orbit
ORDER BY SuccessRatePct DESC;
"""
df_orbit = run_sql(q_orbit, "sql_orbit_success.csv")

# 4) Monthly trend (handle Date that may include time)
q_monthly = """
SELECT
  CASE
    WHEN instr(Date,'-')=5 THEN substr(Date,1,7)                -- 'YYYY-MM'
    ELSE strftime('%Y-%m', Date)                                -- if Date is SQLite date
  END AS YearMonth,
  COUNT(*) AS Flights,
  SUM(class) AS Successes
FROM SPACEXTBL
GROUP BY YearMonth
ORDER BY YearMonth;
"""
df_monthly = run_sql(q_monthly, "sql_monthly_trend.csv")

# 5) Success by payload buckets
q_bins = """
WITH b AS (
  SELECT CASE
           WHEN PayloadMass < 2000 THEN '<2t'
           WHEN PayloadMass < 4000 THEN '2–4t'
           WHEN PayloadMass < 6000 THEN '4–6t'
           WHEN PayloadMass < 8000 THEN '6–8t'
           ELSE '8t+'
         END AS PayloadBin,
         class
  FROM SPACEXTBL
  WHERE PayloadMass IS NOT NULL
)
SELECT PayloadBin AS bin,
       COUNT(*)   AS Flights,
       SUM(class) AS Successes,
       ROUND(100.0*AVG(class),1) AS SuccessRatePct
FROM b
GROUP BY PayloadBin
ORDER BY CASE PayloadBin
  WHEN '<2t' THEN 1 WHEN '2–4t' THEN 2
  WHEN '4–6t' THEN 3 WHEN '6–8t' THEN 4 ELSE 5 END;
"""
df_bins = run_sql(q_bins, "sql_payload_bins.csv")


LaunchSite,Successes,Flights,SuccessRatePct
KSC LC 39A,50,55,90.900000
VAFB SLC 4E,23,28,82.100000
CCSFS SLC 40,72,99,72.700000
Kwajalein Atoll,0,5,0.000000


BoosterVersion,Successes,Flights,SuccessRatePct
None,145,187,77.500000


Orbit,AvgPayloadKg,SuccessRatePct
TLI,None,100.000000
HEO,None,100.000000
HCO,None,100.000000
GEO,None,100.000000
ES-L1,None,100.000000
None,None,100.000000
VLEO,None,94.600000
SSO,None,92.300000
MEO,None,80.000000
ISS,None,72.700000


YearMonth,Flights,Successes
None,187,145


bin,Flights,Successes,SuccessRatePct
